# Research-Quality Figures · plots you could put in a paper

A figure is an argument. A good one makes your result obvious in a few seconds and survives being shrunk into a paper column or printed in black and white. This lab turns the dataset from the previous lab into publication quality figures: the right chart for each claim, a consistent house style, colorblind and grayscale safe colors, and clean vector files you can drop straight into a report or LaTeX.

You need the dataset from **Lab CC · Synthetic Experiment Data** (`~/experimentLab/experiment.csv`). If you have not run that lab yet, the first cell below makes a stand-in dataset so you can still work through the plotting.

Work through it top to bottom. Run every code cell and read what comes back.

## How this notebook works

- **[Notebook cell]** runs here with **Shift+Enter**. Each plotting cell builds one figure and saves it.
- This lab uses `pandas` for the data and `matplotlib` for the plots. Both are already installed in the class image.
- Figures are written to `~/experimentLab/figures/` as both PDF (for print and LaTeX) and PNG (for slides and the web).

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, checkpoints, and the figure helpers.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

### Preflight · check your environment

In [ ]:
preflight([
    check("pandas importable", pythonImportable("pandas"),
          hint="pandas is preinstalled in the class image."),
    check("matplotlib importable", pythonImportable("matplotlib"),
          hint="matplotlib is preinstalled in the class image via ultralytics."),
    check("your home folder is writable", dirExists("~"),
          hint="You need a home folder to save the figures."),
])

---
## Part 1 · Load the experiment

**[Notebook cell]** Load `experiment.csv` from the previous lab. If it is not there, this makes a small stand-in dataset with the same columns so you can still work through the plotting:

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

labDir = Path.home() / "experimentLab"
labDir.mkdir(exist_ok=True)
csvPath = labDir / "experiment.csv"

if not csvPath.exists():
    print("experiment.csv not found - generating a stand-in dataset")
    rng = np.random.default_rng(494)
    rows = []
    for m, med, bw, ut in [("yolov8n", 8, 18, 45), ("yolov8s", 14, 26, 68), ("yolov8m", 26, 38, 88)]:
        lat = np.exp(np.log(med) + 0.35 * rng.standard_normal(30))
        util = np.clip(ut + rng.normal(0, 4, 30), 0, 100)
        power = bw + 0.12 * util + rng.normal(0, 1, 30)
        temp = 40 + 0.6 * power + rng.normal(0, 1.5, 30)
        for i in range(30):
            rows.append(dict(model=m, trial=i + 1,
                latency_ms=round(float(lat[i]), 3), throughput_fps=round(1000 / float(lat[i]), 2),
                power_w=round(float(power[i]), 2), gpu_util_pct=round(float(util[i]), 1),
                temp_c=round(float(temp[i]), 1), energy_mj=round(float(power[i]) * float(lat[i]), 1)))
    pd.DataFrame(rows).to_csv(csvPath, index=False)

%cd ~/experimentLab
data = pd.read_csv(csvPath)
print("loaded", len(data), "rows across", data["model"].nunique(), "models")
data.head()

---
## Part 2 · A consistent house style

Before plotting anything, set one house style so every figure shares the same fonts, sizes, and colors. Consistency is half of what makes a set of figures look professional. `applyHouseStyle` from the lab toolkit sets readable fonts, a colorblind safe color cycle, a light grid, and vector friendly output, and hands back the palette:

In [ ]:
import matplotlib.pyplot as plt

palette = applyHouseStyle()
models = list(data["model"].unique())
colors = {m: palette[i] for i, m in enumerate(models)}
print("house style applied")
print("model colors:", colors)

**[Notebook cell]** Most charts show a summary per model, so compute the mean and standard deviation of every metric once, grouped by model:

In [ ]:
summary = data.groupby("model").agg(["mean", "std"])
summary["latency_ms"]

---
## Part 3 · Match the chart to the claim

Every chart type answers a different question. Here are the ones you will use most, each built from the same dataset and each saved with `saveFigure`.

**[Notebook cell]** A **bar chart with error bars** answers "which model is faster?" The bar height is the mean latency and the black line is one standard deviation, so the reader sees both the value and its uncertainty. Never show a mean without its spread:

In [ ]:
means = data.groupby("model")["latency_ms"].mean().reindex(models)
stds = data.groupby("model")["latency_ms"].std().reindex(models)

fig, ax = plt.subplots()
ax.bar(means.index, means.values, yerr=stds.values, capsize=5,
       color=[colors[m] for m in means.index])
ax.set_xlabel("model")
ax.set_ylabel("latency (ms)")
ax.set_title("Inference latency by model")
saveFigure(fig, "latency_by_model")
plt.show()

**[Notebook cell]** A **box plot** answers "how spread out is it?" It shows the median, the middle 50 percent, and any outliers, which a bar chart hides. Use it when the shape of the data matters, not just its average:

In [ ]:
fig, ax = plt.subplots()
byModel = [data.loc[data["model"] == m, "latency_ms"] for m in models]
bp = ax.boxplot(byModel, patch_artist=True)
ax.set_xticks(range(1, len(models) + 1))
ax.set_xticklabels(models)
for patch, m in zip(bp["boxes"], models):
    patch.set_facecolor(colors[m])
    patch.set_alpha(0.6)
ax.set_xlabel("model")
ax.set_ylabel("latency (ms)")
ax.set_title("Latency distribution by model")
saveFigure(fig, "latency_distribution")
plt.show()

**[Notebook cell]** For latency, the **tail** is often the story: p95 and p99, not the average, decide whether you meet a deadline. A **CDF** (cumulative distribution) shows it directly. Read up from a time on the x axis to see what fraction of runs finished under it:

In [ ]:
fig, ax = plt.subplots()
for i, m in enumerate(models):
    vals = np.sort(data.loc[data["model"] == m, "latency_ms"].values)
    cdf = np.arange(1, len(vals) + 1) / len(vals)
    ax.plot(vals, cdf, marker=FIGURE_MARKERS[i], markevery=5, label=m, color=colors[m])
ax.axhline(0.95, color="gray", linestyle=":", linewidth=1)
ax.text(ax.get_xlim()[1], 0.95, " p95", va="center", fontsize=9, color="gray")
ax.set_xlabel("latency (ms)")
ax.set_ylabel("fraction of runs below")
ax.set_title("Latency CDF (tail behaviour)")
ax.legend(title="model")
saveFigure(fig, "latency_cdf")
plt.show()

**[Notebook cell]** A **scatter plot** answers "what is the tradeoff?" Plot two metrics against each other to reveal a **Pareto front**, the set of best available compromises. Here throughput against power, where up and to the left is better, faster and more efficient:

In [ ]:
fig, ax = plt.subplots()
for i, m in enumerate(models):
    d = data[data["model"] == m]
    ax.scatter(d["power_w"], d["throughput_fps"], label=m, color=colors[m],
               marker=FIGURE_MARKERS[i], alpha=0.7, edgecolor="white")
ax.set_xlabel("power (W)")
ax.set_ylabel("throughput (fps)")
ax.set_title("Throughput vs power")
ax.legend(title="model")
saveFigure(fig, "throughput_vs_power")
plt.show()

**[Notebook cell]** A **line with an error band** answers "how does something change along an axis?" Reusing the idea from the data lab, show how the latency estimate for one model settles as you average more trials. The shaded band is the standard error, and it narrows as evidence accumulates:

In [ ]:
one = data[data["model"] == models[1]]["latency_ms"].values
n = np.arange(1, len(one) + 1)
runningMean = np.cumsum(one) / n
runningSE = np.array([one[:k].std(ddof=1) / np.sqrt(k) if k > 1 else 0.0 for k in n])

fig, ax = plt.subplots()
ax.plot(n, runningMean, color=palette[1])
ax.fill_between(n, runningMean - runningSE, runningMean + runningSE, color=palette[1], alpha=0.25)
ax.set_xlabel("trials averaged")
ax.set_ylabel("estimated mean latency (ms)")
ax.set_title(f"The estimate settles as trials accumulate ({models[1]})")
saveFigure(fig, "estimate_convergence")
plt.show()

When a metric spans orders of magnitude, say latency from 1 ms to 1000 ms, add `ax.set_yscale("log")` so both small and large values stay readable. Our data is narrow, so a linear axis is fine here.

In [ ]:
checkpoint("Part 3 - one figure per claim", [
    check("bar chart saved", fileExists("~/experimentLab/figures/latency_by_model.pdf"),
          hint="Run the bar chart cell."),
    check("distribution plot saved", fileExists("~/experimentLab/figures/latency_distribution.pdf"),
          hint="Run the box plot cell."),
    check("CDF / tail plot saved", fileExists("~/experimentLab/figures/latency_cdf.pdf"),
          hint="Run the CDF cell."),
    check("tradeoff scatter saved", fileExists("~/experimentLab/figures/throughput_vs_power.pdf"),
          hint="Run the scatter cell."),
], successNote="Bar for values, box for spread, CDF for tails, scatter for tradeoffs. Pick the chart that makes your claim obvious.")

---
## Part 4 · Make it publication safe

A figure has to survive real conditions: printed in grayscale, viewed by a colorblind reader, and shrunk to one column. Three rules cover most of it.

- **Encode data twice.** Do not rely on color alone. Pair each color with a distinct **marker** or **line style** so the figure still reads in black and white. The house style already uses a colorblind safe palette; the markers finish the job.
- **Label everything, with units.** Every axis needs a name and a unit, and every multi-series chart needs a legend.
- **Write a caption that states the takeaway**, not just what the axes are.

**[Notebook cell]** Here is the CDF again, built to be fully grayscale safe: each model gets its own line style and marker as well as its color, so it survives black and white printing. Picture it with the colors removed, the lines are still tellable apart:

In [ ]:
fig, ax = plt.subplots()
for i, m in enumerate(models):
    vals = np.sort(data.loc[data["model"] == m, "latency_ms"].values)
    cdf = np.arange(1, len(vals) + 1) / len(vals)
    ax.plot(vals, cdf, label=m, color=colors[m],
            linestyle=FIGURE_LINESTYLES[i % len(FIGURE_LINESTYLES)],
            marker=FIGURE_MARKERS[i], markevery=5)
ax.set_xlabel("latency (ms)")
ax.set_ylabel("fraction of runs below")
ax.set_title("Latency CDF, grayscale safe")
ax.legend(title="model")
saveFigure(fig, "latency_cdf_grayscale")
plt.show()

---
## Part 5 · A multi-panel figure

Papers group related plots into one figure with labelled panels (a), (b), (c). `plt.subplots` lays them out on a grid. Here three metrics side by side, each showing the cost of a bigger model:

In [ ]:
metrics = [("latency_ms", "latency (ms)"), ("power_w", "power (W)"), ("temp_c", "temp (C)")]
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, (col, label) in zip(axes, metrics):
    means = data.groupby("model")[col].mean().reindex(models)
    stds = data.groupby("model")[col].std().reindex(models)
    ax.bar(means.index, means.values, yerr=stds.values, capsize=4,
           color=[colors[m] for m in means.index])
    ax.set_ylabel(label)
    ax.tick_params(axis="x", rotation=20)
for letter, ax in zip("abc", axes):
    ax.set_title(f"({letter})", loc="left", fontweight="bold")
fig.suptitle("Cost of a bigger model: latency, power, and heat")
fig.tight_layout()
saveFigure(fig, "model_cost_panels")
plt.show()

---
## Part 6 · Export for your report

`saveFigure` already wrote each figure as both **PDF** (vector, for LaTeX and print, stays sharp at any size) and **PNG** (raster, for slides and the web). List everything you produced:

In [ ]:
for p in sorted((labDir / "figures").iterdir()):
    print(p.name)

To include a figure in a LaTeX paper, use the PDF and write a caption that states the finding, not just the axes:

    \begin{figure}[t]
      \centering
      \includegraphics[width=\columnwidth]{figures/latency_by_model.pdf}
      \caption{yolov8m costs roughly 3x the latency of yolov8n for its extra accuracy.}
      \label{fig:latency}
    \end{figure}

For a slide or a Markdown report, point at the PNG instead.

In [ ]:
checkpoint("Part 6 - figures exported", [
    check("multi-panel PDF exists", fileExists("~/experimentLab/figures/model_cost_panels.pdf"),
          hint="Run the multi-panel cell."),
    check("PNG copies exist too", fileExists("~/experimentLab/figures/model_cost_panels.png"),
          hint="saveFigure writes both PDF and PNG."),
    check("grayscale-safe figure saved", fileExists("~/experimentLab/figures/latency_cdf_grayscale.pdf"),
          hint="Run the grayscale CDF cell in Part 4."),
], successNote="Vector PDFs for print, PNGs for slides, colorblind and grayscale safe, captioned. These are figures you could submit.")

### Lab scorecard

In [ ]:
labSummary("Research-Quality Figures")

---

That completes the on-ramp. You can drive the machine (**AA**), work in Python (**BB**), design a reproducible experiment and record clean data (**CC**), and turn that data into figures worth publishing (**DD**). Everything from here builds on these.